# QM 640 Capstone — Step 3g: Reclassification Diagnostic & Independent Recode Reset

**Why this notebook exists:** `[98] Manual Population 1` and `[99] Manual Population 2` were run on `screening_recode_sample.csv` and `screening_TO_REVIEW.csv` respectively, *before* `03f`. Their AI-keyword matcher does plain substring matching, so a bare `"ai"` keyword matches inside ordinary words like "maintain," "certain," "available," and even the pipeline's own "FAILED" fetch-error text. That means:

1. `is_genuine_ai_event` / `announcement_type` currently in `screening_worksheet.csv` may contain script-driven false positives, not confirmed human judgments.
2. `recoder_is_genuine_ai_event` / `recoder_announcement_type` in `screening_recode_sample.csv` were generated by the *same deterministic logic* — so any Cohen's kappa computed against it is not a real inter-rater check.

This notebook does two things, and does **not** overwrite `screening_worksheet.csv` itself:

- **Part A** — re-classifies every row in `screening_TO_REVIEW.csv` with a corrected, word-boundary matcher, compares it to what's currently recorded, and writes `screening_REVIEW_PRIORITY.csv` — a short, ranked list of exactly which rows need a human to actually look at them, instead of all ~500+.
- **Part B** — resets `screening_recode_sample.csv`'s `recoder_*` columns to blank (keeping the same sampled rows and their `text_snippet`), so a genuinely independent second coder can fill them in without seeing any prior answer.

**Run this after `03e_fetch_snippets` and instead of `[98]`/`[99]`. Run `03f` and `03`'s Part B (kappa) after you've finished the manual review this notebook sets up.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
REVIEW_FILE = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")
REVIEW_PRIORITY_FILE = os.path.join(RAW_DIR, "screening_REVIEW_PRIORITY.csv")
RECODE_PRIORITY_FILE = os.path.join(RAW_DIR, "screening_RECODE_PRIORITY.csv")

## Cell 4 — Corrected classifier (word-boundary matching)

Same keyword lists as `[98]`/`[99]`, but matching is now done with `\b...\b` regex boundaries, so `"ai"` only matches the standalone word "AI," not the substring inside "maintain," "certain," "available," "FAILED," etc. Also flags **low-confidence** matches - rows where the *only* reason for a `Y` is a single generic keyword (bare "ai", or an M&A keyword like "purchase"/"agreement" that's common in unrelated 8-K boilerplate) - since those are still worth a human glance even when the word-boundary fix alone doesn't flip the result.

In [ ]:
import re
import pandas as pd

AI_KEYWORDS = [
    "artificial intelligence", "ai", "generative ai", "genai", "machine learning",
    "deep learning", "large language model", "language model", "foundation model", "llm",
    "gpt", "copilot", "neural network", "computer vision", "predictive ai", "autonomous ai",
    "ai assistant", "chatbot", "ai platform", "ai software", "ai infrastructure", "ai chip",
    "ai accelerator",
]
PARTNERSHIP_KEYWORDS = [
    "partner", "partners", "partnered", "partnership", "collaborate", "collaboration",
    "alliance", "strategic alliance", "joint venture", "jointly", "agreement",
    "memorandum of understanding", "mou", "teamed up", "working with",
]
RD_KEYWORDS = [
    "launch", "launched", "introduce", "introduced", "release", "released", "develop",
    "developed", "developing", "research", "innovation", "innovative", "prototype", "patent",
    "roadmap", "platform", "solution", "tool", "engine", "framework", "service", "capability",
    "model", "assistant", "copilot", "training", "inference", "gpu", "chip", "processor",
]
MA_KEYWORDS = [
    "acquire", "acquired", "acquisition", "buy", "bought", "purchase", "purchased",
    "takeover", "merge", "merged", "merger", "invests in", "investment in", "stake in",
]

# Keywords too generic to trust on their own, even with word boundaries - a hit on
# ONLY these (no more specific keyword also matching) gets flagged for a spot check.
GENERIC_AI_KEYWORDS = {"ai"}
GENERIC_MA_KEYWORDS = {"purchase", "purchased", "buy", "bought", "agreement", "stake in"}


def preprocess(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).lower())


def matched_keywords(text, keywords):
    """Word-boundary match - returns the list of keywords that actually hit."""
    return [k for k in keywords if re.search(rf"\b{re.escape(k)}\b", text)]


def classify_fixed(snippet):
    text = preprocess(snippet)
    if text == "":
        return "N", "", [], False

    ai_hits = matched_keywords(text, AI_KEYWORDS)
    if not ai_hits:
        return "N", "", [], False

    low_confidence = set(ai_hits).issubset(GENERIC_AI_KEYWORDS)

    ma_hits = matched_keywords(text, MA_KEYWORDS)
    if ma_hits:
        if set(ma_hits).issubset(GENERIC_MA_KEYWORDS):
            low_confidence = True
        return "Y", "M&A", ai_hits + ma_hits, low_confidence

    partner_hits = matched_keywords(text, PARTNERSHIP_KEYWORDS)
    if partner_hits:
        return "Y", "partnership", ai_hits + partner_hits, low_confidence

    rd_hits = matched_keywords(text, RD_KEYWORDS)
    if rd_hits:
        return "Y", "R&D", ai_hits + rd_hits, low_confidence

    # AI mentioned, no more specific type keyword matched - low confidence by definition
    return "Y", "R&D", ai_hits, True


print("Corrected classifier loaded.")

## Cell 5 — Part A: Diff the review file against the corrected classifier

In [ ]:
review = pd.read_csv(REVIEW_FILE)
print(f"Loaded {len(review)} rows from {REVIEW_FILE}")

fixed_results = review["text_snippet"].apply(classify_fixed)
review["fixed_is_genuine_ai_event"] = fixed_results.apply(lambda r: r[0])
review["fixed_announcement_type"] = fixed_results.apply(lambda r: r[1])
review["matched_keywords"] = fixed_results.apply(lambda r: ", ".join(r[2]))
review["low_confidence_match"] = fixed_results.apply(lambda r: r[3])

current_y = review["is_genuine_ai_event"].astype(str).str.strip().str.upper()
current_type = review["announcement_type"].astype(str).str.strip()

disagree_y = current_y != review["fixed_is_genuine_ai_event"]
disagree_type = (current_y == "Y") & (current_type != review["fixed_announcement_type"])
disagreement = disagree_y | disagree_type

def priority_label(row_disagree, row_low_conf):
    if row_disagree:
        return "HIGH - classifier disagreement"
    if row_low_conf:
        return "MEDIUM - single generic keyword only"
    return None

review["review_priority"] = [
    priority_label(d, lc) for d, lc in zip(disagreement, review["low_confidence_match"])
]

priority_rows = review[review["review_priority"].notna()].copy()
priority_rows = priority_rows.sort_values("review_priority")

out_cols = [
    "accession_no", "company_name", "file_date", "text_snippet",
    "is_genuine_ai_event", "announcement_type",              # current (script-derived) values
    "fixed_is_genuine_ai_event", "fixed_announcement_type",  # corrected classifier's opinion
    "matched_keywords", "review_priority",
    "filing_url",
]
priority_rows = priority_rows[[c for c in out_cols if c in priority_rows.columns]]

# Blank columns for YOUR final human decision - fill these in, not the ones above
priority_rows["final_is_genuine_ai_event"] = ""
priority_rows["final_announcement_type"] = ""

priority_rows.to_csv(REVIEW_PRIORITY_FILE, index=False)

n_high = (review["review_priority"] == "HIGH - classifier disagreement").sum()
n_med = (review["review_priority"] == "MEDIUM - single generic keyword only").sum()
n_current_y = (current_y == "Y").sum()

print(f"\nCurrently marked Y (genuine AI event): {n_current_y} / {len(review)}")
print(f"HIGH priority (classifier disagrees - likely bug-driven): {n_high}")
print(f"MEDIUM priority (single generic keyword only - spot check): {n_med}")
print(f"Total rows needing a human look: {n_high + n_med} / {len(review)} "
      f"({(n_high + n_med) / len(review):.1%})")
print(f"\nSaved -> {REVIEW_PRIORITY_FILE}")
priority_rows.head()

## Cell 6 — Part A (diagnostic only): same diff, run against the recode sample

This is informational only — the recode sample itself gets reset to blank in Part B regardless, since its `recoder_*` columns aren't trustworthy either way (same script, same bug). This just shows you how contaminated the *old* recode results were.

In [ ]:
recode = pd.read_csv(RECODE_FILE)
print(f"Loaded {len(recode)} rows from {RECODE_FILE}")

fixed_results_r = recode["text_snippet"].apply(classify_fixed)
recode["fixed_is_genuine_ai_event"] = fixed_results_r.apply(lambda r: r[0])
recode["fixed_announcement_type"] = fixed_results_r.apply(lambda r: r[1])
recode["matched_keywords"] = fixed_results_r.apply(lambda r: ", ".join(r[2]))

old_y = recode.get("recoder_is_genuine_ai_event", pd.Series(dtype=str)).astype(str).str.strip().str.upper()
if len(old_y) == len(recode):
    disagree_r = old_y != recode["fixed_is_genuine_ai_event"]
    print(f"Old recoder_is_genuine_ai_event disagreed with corrected classifier on "
          f"{disagree_r.sum()}/{len(recode)} rows ({disagree_r.mean():.1%}) - "
          f"illustrates why this sample needs a real independent pass, not a reused script.")

recode[["accession_no", "company_name", "file_date", "text_snippet",
        "fixed_is_genuine_ai_event", "fixed_announcement_type", "matched_keywords"]].to_csv(
    RECODE_PRIORITY_FILE, index=False)
print(f"Saved diagnostic -> {RECODE_PRIORITY_FILE}")

## Cell 7 — Part B: reset the recode sample for a genuinely independent coder

Keeps the same sampled rows (so the kappa comparison is apples-to-apples with the primary review) and their `text_snippet`, but blanks the `recoder_*` columns and drops any prior-answer columns, so whoever codes this next sees nothing but the raw text - no hint of what the script said or what you said in your first pass.

In [ ]:
recode_blind = recode[["accession_no", "company_name", "file_date", "text_snippet"]].copy()
recode_blind["recoder_announcement_type"] = ""
recode_blind["recoder_is_genuine_ai_event"] = ""

recode_blind.to_csv(RECODE_FILE, index=False)
print(f"Reset -> {RECODE_FILE} ({len(recode_blind)} rows, recoder columns blanked)")
print("\nHand this file to your independent coder (a different person, or yourself working "
      "blind days later without looking at screening_REVIEW_PRIORITY.csv or your earlier "
      "answers). They should NOT see is_genuine_ai_event/announcement_type from anywhere else.")

## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_REVIEW_PRIORITY.csv"
!git -C {BASE_DIR} add "data/raw/screening_RECODE_PRIORITY.csv"
!git -C {BASE_DIR} add "data/raw/screening_recode_sample.csv"
!git -C {BASE_DIR} commit -m "Step 3g: diagnose classifier disagreements, reset recode sample for a blind independent pass"
!git -C {BASE_DIR} push

## >>> STOP HERE — manual step, outside this notebook <<<

1. Open `screening_REVIEW_PRIORITY.csv`. For every row, read `text_snippet` (click `filing_url` only if it's ambiguous or cut off) and fill in `final_is_genuine_ai_event` (Y/N) and `final_announcement_type` (partnership/R&D/M&A). This list is short - only rows the corrected classifier disagrees with, or where the only evidence was a single generic keyword - not all ~500+ rows.
2. Have an independent person (or yourself, blind, on a different day) fill in `recoder_is_genuine_ai_event` / `recoder_announcement_type` in the reset `screening_recode_sample.csv`, using only `text_snippet` - no access to your answers or the classifier's suggestions.
3. Push both completed files back to the repo.
4. Before running `03f`, merge `screening_REVIEW_PRIORITY.csv`'s `final_*` decisions back into `screening_TO_REVIEW.csv` (overwrite `is_genuine_ai_event`/`announcement_type` for the rows you reviewed; leave every other row's existing value untouched, since it wasn't flagged). Then run `03f_merge_review.ipynb` as normal, followed by `03`'s Part B (Cohen's kappa) - which will now be a real inter-rater check for the first time.